In [0]:
# Objective: Build one-row-per-user Gold feature tables for
# both the training cohort (with label) and scoring cohort
# (no label), using the same feature logic.
#
# Key design decision from Step 2 EDA:
#   - Training cohort: must use checkpoint-anchored reference
#     point (Feb-expiry transaction), NOT latest transaction,
#     to avoid label leakage.
#   - Scoring cohort: no leakage risk, use latest transaction
#     as of March 31 reference date.


BRONZE = "churn_project.bronze"
GOLD   = "churn_project.gold"

import pyspark.sql.functions as F
from pyspark.sql.window import Window

df_train       = spark.table(f"{BRONZE}.train_v2")
df_members     = spark.table(f"{BRONZE}.members")
df_txn_train   = spark.table(f"{BRONZE}.transactions_train")
df_txn_score   = spark.table(f"{BRONZE}.transactions_score")
df_logs_train  = spark.table(f"{BRONZE}.user_logs_train")
df_logs_score  = spark.table(f"{BRONZE}.user_logs_score")

REFERENCE_DATE_TRAIN = F.lit("2017-02-28").cast("date")
REFERENCE_DATE_SCORE = F.lit("2017-03-31").cast("date")


In [0]:
# Step A: find each user's Feb-2017-expiry transaction (the real checkpoint)
feb_checkpoint_txns = (
    df_txn_train
    .filter(F.date_format("membership_expire_date", "yyyy-MM") == "2017-02")
)

# Tiebreak: if a user has multiple Feb-expiry transactions (plan changes),
# take the one with the LATEST transaction_date (most recent decision)
window_feb = Window.partitionBy("msno").orderBy(F.col("transaction_date").desc())

checkpoint_txn = (
    feb_checkpoint_txns
    .withColumn("rn", F.row_number().over(window_feb))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumn("has_feb_checkpoint", F.lit(1))
)

print(f"Users with a clean Feb-checkpoint transaction: {checkpoint_txn.count():,}")

# Step B: for users WITHOUT a Feb-checkpoint, fall back to their
# EARLIEST transaction in the window (best available proxy for
# "status going into the decision period")
users_with_checkpoint = checkpoint_txn.select("msno")

window_earliest = Window.partitionBy("msno").orderBy(F.col("transaction_date").asc())

fallback_txn = (
    df_txn_train
    .join(users_with_checkpoint, "msno", "left_anti")   # only users WITHOUT a checkpoint
    .withColumn("rn", F.row_number().over(window_earliest))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .withColumn("has_feb_checkpoint", F.lit(0))
)

print(f"Users using fallback (earliest transaction): {fallback_txn.count():,}")

# Combine: this is now each user's single reference transaction
reference_txn_train = checkpoint_txn.unionByName(fallback_txn)

print(f"\nTotal reference transactions (should equal transacting users): {reference_txn_train.count():,}")
print(f"(Recall: 936,986 labeled users have at least one transaction; "
      f"{970960 - 936986:,} have none and will have null transaction features)")

Users with a clean Feb-checkpoint transaction: 795,559
Users using fallback (earliest transaction): 141,427

Total reference transactions (should equal transacting users): 936,986
(Recall: 936,986 labeled users have at least one transaction; 33,974 have none and will have null transaction features)


In [0]:
# Subscription features computed directly from each user's single
# reference transaction (checkpoint or fallback).

subscription_features_train = (
    reference_txn_train
    .select(
        "msno",
        "has_feb_checkpoint",
        F.col("is_auto_renew").alias("auto_renew_at_checkpoint"),
        F.col("payment_plan_days").alias("plan_days_at_checkpoint"),
        F.col("actual_amount_paid").alias("amount_paid_at_checkpoint"),
        F.col("plan_list_price").alias("list_price_at_checkpoint"),
        F.when(F.col("actual_amount_paid") == 0, 1).otherwise(0).alias("is_zero_paid_at_checkpoint"),
        F.col("membership_expire_date").alias("checkpoint_expire_date"),
        "transaction_date"
    )
)

print(f"Subscription features built for {subscription_features_train.count():,} users")
subscription_features_train.show(5, truncate=False)

Subscription features built for 936,986 users
+--------------------------------------------+------------------+------------------------+-----------------------+-------------------------+------------------------+--------------------------+----------------------+----------------+
|msno                                        |has_feb_checkpoint|auto_renew_at_checkpoint|plan_days_at_checkpoint|amount_paid_at_checkpoint|list_price_at_checkpoint|is_zero_paid_at_checkpoint|checkpoint_expire_date|transaction_date|
+--------------------------------------------+------------------+------------------------+-----------------------+-------------------------+------------------------+--------------------------+----------------------+----------------+
|+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=|1                 |1                       |30                     |99                       |99                      |0                         |2017-02-15            |2017-01-15      |
|++/9R3sX37CjxbY/AaGvb

In [0]:
# These look across ALL transactions in the window per user — different
# from the single reference-transaction features above.

history_features_train = (
    df_txn_train
    .groupBy("msno")
    .agg(
        F.max("is_cancel").alias("ever_cancelled"),
        F.max(F.when(F.col("actual_amount_paid") == 0, 1).otherwise(0)).alias("ever_zero_paid"),
        F.count("*").alias("transaction_count"),
        F.countDistinct("payment_method_id").alias("payment_method_count"),
        F.avg("actual_amount_paid").alias("avg_amount_paid"),
        F.min("transaction_date").alias("earliest_txn_in_window"),
        F.max("transaction_date").alias("latest_txn_in_window")
    )
)

print(f"History features built for {history_features_train.count():,} users")
history_features_train.show(5, truncate=False)

History features built for 936,986 users
+--------------------------------------------+--------------+--------------+-----------------+--------------------+---------------+----------------------+--------------------+
|msno                                        |ever_cancelled|ever_zero_paid|transaction_count|payment_method_count|avg_amount_paid|earliest_txn_in_window|latest_txn_in_window|
+--------------------------------------------+--------------+--------------+-----------------+--------------------+---------------+----------------------+--------------------+
|zSF+lYWjWOdRM8PZHpj3c/V7ebJTp1YXma504ubO0kg=|0             |0             |2                |1                   |99.0           |2017-01-02            |2017-02-02          |
|LYAw9LK7vipq0EaKc43zRAaobZLkkubieJnKne5NOYg=|0             |0             |2                |1                   |149.0          |2017-01-16            |2017-02-16          |
|W7LGWt+Ceqs7m8SIjfoAfV+kr629sma0qdpjSCEWWHU=|1             |0             |3  

In [0]:
tenure_features = (
    df_members
    .select("msno", "registration_init_time", "city", "gender", "registered_via", "bd")
    .withColumn(
        "tenure_days_raw",
        F.datediff(REFERENCE_DATE_TRAIN, F.col("registration_init_time"))
    )
    # Clip negative tenure to 0, per our EDA finding (0.11% of labeled cohort affected)
    .withColumn("tenure_days", F.greatest(F.col("tenure_days_raw"), F.lit(0)))
    .withColumn(
        "tenure_bucket",
        F.when(F.col("tenure_days") < 182, "< 6 months")
         .when(F.col("tenure_days") < 365, "6-12 months")
         .when(F.col("tenure_days") < 1095, "1-3 years")
         .when(F.col("tenure_days") < 2190, "3-6 years")
         .otherwise("6+ years")
    )
    # Clean bd (age) — null out clearly invalid values, per Kaggle's own warning
    .withColumn(
        "age_cleaned",
        F.when((F.col("bd") < 5) | (F.col("bd") > 100), None).otherwise(F.col("bd"))
    )
    # Gender as 3-way categorical, no imputation — per EDA finding (non-random nulls)
    .withColumn(
        "gender_clean",
        F.when(F.col("gender").isNull(), "unknown").otherwise(F.col("gender"))
    )
    .drop("tenure_days_raw", "bd", "gender")
)

print(f"Tenure/member features built for {tenure_features.count():,} users")
tenure_features.select("msno", "tenure_days", "tenure_bucket", "age_cleaned", "gender_clean").show(5, truncate=False)

Tenure/member features built for 6,769,473 users
+--------------------------------------------+-----------+-------------+-----------+------------+
|msno                                        |tenure_days|tenure_bucket|age_cleaned|gender_clean|
+--------------------------------------------+-----------+-------------+-----------+------------+
|yzFPxbifmBAfOyAXEK8c1YpbzTIybe7mvqsIThkk/14=|822        |1-3 years    |31         |female      |
|tMgZwcUAyp7imrUEY9UojHY+rFTuvImgGGc2/3VH7YY=|822        |1-3 years    |50         |female      |
|Csuspz3nYuIEEPs0LDYrEPA/rv2RQLxuXmwa9gzWXt8=|821        |1-3 years    |NULL       |female      |
|nPgDtm5jYl8yGjiaXTnWSnkxDZ1aZPHUaEaQFQQ6w5w=|821        |1-3 years    |NULL       |unknown     |
|EwKWNUEcvhjVp2v2iN2rFpuQ35faWtFM5qUzes01ikk=|821        |1-3 years    |22         |male        |
+--------------------------------------------+-----------+-------------+-----------+------------+
only showing top 5 rows


In [0]:
engagement_features_train = (
    df_logs_train
    .groupBy("msno")
    .agg(
        F.count("*").alias("active_days"),
        F.sum("total_secs").alias("total_listening_secs"),
        F.sum(F.col("num_25") + F.col("num_50") + F.col("num_75") + F.col("num_985") + F.col("num_100")).alias("total_streams"),
        F.sum("num_100").alias("total_completed_streams"),
        F.avg("num_unq").alias("avg_unique_songs")
    )
    .withColumn("avg_daily_streams", F.col("total_streams") / F.col("active_days"))
    .withColumn("avg_listening_secs_per_day", F.col("total_listening_secs") / F.col("active_days"))
    .withColumn("completion_rate", F.col("total_completed_streams") / F.col("total_streams"))
    .withColumn(
        "engagement_bucket",
        F.when(F.col("avg_daily_streams") < 20.60, "Light")
         .when(F.col("avg_daily_streams") < 35.33, "Medium")
         .otherwise("Heavy")
    )
)

print(f"Engagement features built for {engagement_features_train.count():,} users (have log activity)")
engagement_features_train.show(5, truncate=False)

Engagement features built for 781,835 users (have log activity)
+--------------------------------------------+-----------+--------------------+-------------+-----------------------+------------------+------------------+--------------------------+------------------+-----------------+
|msno                                        |active_days|total_listening_secs|total_streams|total_completed_streams|avg_unique_songs  |avg_daily_streams |avg_listening_secs_per_day|completion_rate   |engagement_bucket|
+--------------------------------------------+-----------+--------------------+-------------+-----------------------+------------------+------------------+--------------------------+------------------+-----------------+
|odOsJf9kQXCEU9dJAN7/U9HRry77Z9nF8GwkEW6bUh8=|59         |307184.0690000001   |1370         |1193                   |19.54237288135593 |23.220338983050848|5206.509644067798         |0.8708029197080291|Medium           |
|n0zpf7S8oGH7HLvIy6sNXYlrBtfTNrKr0YhfvSEEjPc=|41        

In [0]:
# Start from train_v2 (the label-defining population) and left-join
# everything else, so we end up with exactly 970,960 rows — one per
# labeled user, with nulls where data genuinely doesn't exist.

gold_user_features_train = (
    df_train
    .join(subscription_features_train, "msno", "left")
    .join(history_features_train.drop("avg_amount_paid"), "msno", "left")  # avoid dup with subscription's amount fields
    .join(tenure_features, "msno", "left")
    .join(engagement_features_train, "msno", "left")
)

# Zero-impute engagement for passive subscribers (txn exists, but no logs).
# True nulls remain for users with no data at all (33,974 label-only users).
engagement_cols_to_zero_impute = [
    "active_days", "total_listening_secs", "total_streams",
    "total_completed_streams", "avg_daily_streams",
    "avg_listening_secs_per_day", "completion_rate"
]

for c in engagement_cols_to_zero_impute:
    gold_user_features_train = gold_user_features_train.withColumn(
        c,
        F.when(
            F.col("transaction_count").isNotNull() & F.col(c).isNull(),  # has txn but no logs
            F.lit(0)
        ).otherwise(F.col(c))
    )

# For passive subscribers, engagement_bucket should be "Light" (zero activity), not null
gold_user_features_train = gold_user_features_train.withColumn(
    "engagement_bucket",
    F.when(
        F.col("transaction_count").isNotNull() & F.col("engagement_bucket").isNull(),
        F.lit("Light")
    ).otherwise(F.col("engagement_bucket"))
)

print(f"Gold training feature table: {gold_user_features_train.count():,} rows")
print(f"(Should equal 970,960 — one row per labeled user)")

print("\nNull rate check on key features:")
gold_user_features_train.select(
    "has_feb_checkpoint", "auto_renew_at_checkpoint", "ever_cancelled",
    "ever_zero_paid", "tenure_days", "avg_daily_streams"
).summary("count").show()

Gold training feature table: 970,960 rows
(Should equal 970,960 — one row per labeled user)

Null rate check on key features:
+-------+------------------+------------------------+--------------+--------------+-----------+-----------------+
|summary|has_feb_checkpoint|auto_renew_at_checkpoint|ever_cancelled|ever_zero_paid|tenure_days|avg_daily_streams|
+-------+------------------+------------------------+--------------+--------------+-----------+-----------------+
|  count|            936986|                  936986|        936986|        936986|     860967|           964946|
+-------+------------------+------------------------+--------------+--------------+-----------+-----------------+



In [0]:
# Investigate: who has non-null avg_daily_streams but NO transaction record?
# This would indicate either a logic bug in the zero-impute step, or a
# legitimate case we haven't accounted for (e.g. user has logs but no
# transactions in our window).

unexpected_non_null = (
    gold_user_features_train
    .filter(F.col("avg_daily_streams").isNotNull() & F.col("transaction_count").isNull())
)

print(f"Users with non-null avg_daily_streams but NO transaction record: {unexpected_non_null.count():,}")

unexpected_non_null.select(
    "msno", "transaction_count", "active_days", "avg_daily_streams", "engagement_bucket"
).show(10, truncate=False)

# Cross-check: how many labeled users have log activity but NOT transaction activity?
# (We'd expect this to explain the gap)
log_only_users = (
    df_train
    .join(df_logs_train.select("msno").distinct(), "msno", "inner")
    .join(df_txn_train.select("msno").distinct(), "msno", "left_anti")
)
print(f"\nLabeled users with logs but NO transactions in window: {log_only_users.count():,}")

Users with non-null avg_daily_streams but NO transaction record: 27,960
+--------------------------------------------+-----------------+-----------+------------------+-----------------+
|msno                                        |transaction_count|active_days|avg_daily_streams |engagement_bucket|
+--------------------------------------------+-----------------+-----------+------------------+-----------------+
|DiqvO3FTsfnKoUwOh8EAEkUtT/W2mET+Q5ynQp4y8ZI=|NULL             |55         |18.836363636363636|Light            |
|ut904CSNdFPmxVQdCC0aI2qUZrxGea34ho9iy7oNaUg=|NULL             |59         |64.2542372881356  |Heavy            |
|PiWI5fBUN/eFyEAmYKgFzL0fP10apgxgXIudtheKqv0=|NULL             |47         |31.53191489361702 |Medium           |
|/wk33k5QFSkvIhIkkfpom3hwAAZmUoMXdUF2z1NLoVA=|NULL             |35         |20.114285714285714|Light            |
|PU9dfhDKlmt/wkW2hRbLO8Rn+8gZZ1Z5oQFAEA+1vjM=|NULL             |3          |17.0              |Light            |
|jHV54J41yVan9NQ

## Third population: log-only users (no transaction record)

In addition to the populations already identified (transacting users with a
Feb-checkpoint, transacting users without one), there is a third group:
**27,960 labeled users have log activity but no transaction record at all**
in our 60-day window. These are likely users whose relevant subscription
transaction fell just outside our Jan 1 cutoff, while their listening
activity (which has its own date range) still falls inside it.

For these users, `transaction_count` and all subscription-derived features
remain genuinely null (we have no transaction to anchor them to), while their
engagement features are real, non-imputed values from actual listening logs.
This is intentional and correct — these aren't passive subscribers (we don't
even know their subscription status), so zero-imputing engagement for them
would be wrong; their non-null engagement values are kept as-is.

**Updated population breakdown of the 970,960 labeled users:**
- 795,559 (81.9%): clean Feb-checkpoint transaction
- 141,427 (14.6%): transaction exists, no Feb-checkpoint (fallback used)
- 27,960 (2.9%): log activity only, no transaction in window

In [0]:
# Full population reconciliation
print("Population breakdown check:")
print(f"  Total labeled users                          : {970960:,}")
print(f"  With Feb-checkpoint                          : 795,559")
print(f"  With fallback (txn, no Feb-checkpoint)       : 141,427")
print(f"  Log-only (no txn at all)                     : 27,960")
print(f"  Neither txn nor log                          : 6,014")
print(f"  Sum check                                    : {795559+141427+27960+6014:,}")

# Quick churn-rate spot check on our two strongest predictive features,
# computed directly from the Gold table — should match EDA findings exactly
print("\nSpot check — auto_renew_at_checkpoint vs churn (should match EDA: ~3.81% vs ~29.48%):")
gold_user_features_train.groupBy("auto_renew_at_checkpoint").agg(
    F.count("*").alias("total"),
    F.round(F.avg("is_churn") * 100, 2).alias("churn_rate_pct")
).orderBy("auto_renew_at_checkpoint").show()

print("Spot check — ever_cancelled vs churn (should match EDA: ~5.59% vs ~27.21%):")
gold_user_features_train.groupBy("ever_cancelled").agg(
    F.count("*").alias("total"),
    F.round(F.avg("is_churn") * 100, 2).alias("churn_rate_pct")
).orderBy("ever_cancelled").show()

Population breakdown check:
  Total labeled users                          : 970,960
  With Feb-checkpoint                          : 795,559
  With fallback (txn, no Feb-checkpoint)       : 141,427
  Log-only (no txn at all)                     : 27,960
  Neither txn nor log                          : 6,014
  Sum check                                    : 970,960

Spot check — auto_renew_at_checkpoint vs churn (should match EDA: ~3.81% vs ~29.48%):
+------------------------+------+--------------+
|auto_renew_at_checkpoint| total|churn_rate_pct|
+------------------------+------+--------------+
|                    NULL| 33974|         84.29|
|                       0| 90517|         28.81|
|                       1|846469|          3.85|
+------------------------+------+--------------+

Spot check — ever_cancelled vs churn (should match EDA: ~5.59% vs ~27.21%):
+--------------+------+--------------+
|ever_cancelled| total|churn_rate_pct|
+--------------+------+--------------+
|        

## Very Strong churn signal: complete absence of activity

Users with NO transaction or log record anywhere in our 60-day window
(33,974 users, 3.5% of labeled cohort) churn at **84.29%**-  by far the
strongest signal found in this project, dwarfing auto-renew (29.48% vs
3.81%), cancellation (27.21% vs 5.59%), and zero-payment history (35.27%
vs 6.11%).

**Interpretation:** these are very likely users who had already effectively
lapsed before our window began — no subscription activity, no listening
activity, nothing recent to anchor any feature to. Their February churn
label reflects a decision that, behaviorally, had already been made before
January even started.

**Action:** added an explicit `has_any_activity` (0/1) feature so the model
sees this directly, rather than inferring it indirectly through nulls
scattered across several columns. This is expected to be a top feature in
Step 4.

In [0]:
# Add an explicit flag for this powerful signal: total absence of activity
gold_user_features_train = gold_user_features_train.withColumn(
    "has_any_activity",
    F.when(
        F.col("transaction_count").isNotNull() | F.col("active_days").isNotNull(),
        1
    ).otherwise(0)
)

print("Verify the new flag captures this correctly:")
gold_user_features_train.groupBy("has_any_activity").agg(
    F.count("*").alias("total"),
    F.round(F.avg("is_churn") * 100, 2).alias("churn_rate_pct")
).show()

Verify the new flag captures this correctly:
+----------------+------+--------------+
|has_any_activity| total|churn_rate_pct|
+----------------+------+--------------+
|               1|964946|          8.56|
|               0|  6014|         78.15|
+----------------+------+--------------+



In [0]:
# Three-way breakdown: transacting / log-only / fully inactive
print("Three-way churn rate breakdown:")
gold_user_features_train.withColumn(
    "activity_group",
    F.when(F.col("transaction_count").isNotNull(), "Has transaction")
     .when(F.col("active_days").isNotNull(), "Log-only (no transaction)")
     .otherwise("No activity at all")
).groupBy("activity_group").agg(
    F.count("*").alias("total"),
    F.round(F.avg("is_churn") * 100, 2).alias("churn_rate_pct")
).orderBy(F.col("total").desc()).show(truncate=False)

Three-way churn rate breakdown:
+-------------------------+------+--------------+
|activity_group           |total |churn_rate_pct|
+-------------------------+------+--------------+
|Has transaction          |936986|6.26          |
|Log-only (no transaction)|27960 |85.61         |
|No activity at all       |6014  |78.15         |
+-------------------------+------+--------------+



In [0]:
# Correct the feature: the real signal is transaction visibility,
# not activity in general (log-only users churn even MORE than
# fully-inactive users, so "any activity" was the wrong framing)

gold_user_features_train = gold_user_features_train.withColumn(
    "has_visible_transaction",
    F.when(F.col("transaction_count").isNotNull(), 1).otherwise(0)
)

print("Verify corrected flag:")
gold_user_features_train.groupBy("has_visible_transaction").agg(
    F.count("*").alias("total"),
    F.round(F.avg("is_churn") * 100, 2).alias("churn_rate_pct")
).show()

# Drop the incorrectly-framed flag from before
gold_user_features_train = gold_user_features_train.drop("has_any_activity")

Verify corrected flag:
+-----------------------+------+--------------+
|has_visible_transaction| total|churn_rate_pct|
+-----------------------+------+--------------+
|                      1|936986|          6.26|
|                      0| 33974|         84.29|
+-----------------------+------+--------------+



### Correction: the real signal is transaction visibility, not "any activity"

Initial framing ("no activity at all" → 84% churn) was imprecise. A three-way
breakdown reveals something sharper:

| Group | Users | Churn rate |
|---|---|---|
| Has a visible transaction | 936,986 | 6.26% |
| Log-only (listened, but no visible transaction) | 27,960 | **85.61%** |
| No activity at all | 6,014 | 78.15% |

The log-only group churns at a *higher* rate than the fully-inactive group- 
counter to a naive "more activity = more engaged = less likely to churn"
assumption. This confirms that **transaction visibility, not engagement, is
the dominant signal**: a user who kept streaming on a subscription we can't
see being renewed is just as (or more) likely to have actually churned as
someone who went completely dark. This is consistent with our earlier finding
that engagement does not predict churn-  it's reinforced here at the extreme.

**Corrected feature:** `has_visible_transaction` (1/0), replacing the earlier
`has_any_activity` framing. This cleanly separates the ~6.26% churn group
(936,986 users with a transaction) from the ~84% churn group (33,974 users
without one).

In [0]:
# Drop intermediate/redundant columns before writing Gold.

gold_user_features_train_final = gold_user_features_train.select(
    "msno",
    "is_churn",

    # Population / data quality flags
    "has_feb_checkpoint",
    "has_visible_transaction",

    # Subscription features (primary churn predictors per EDA)
    "auto_renew_at_checkpoint",
    "ever_cancelled",
    "ever_zero_paid",
    "is_zero_paid_at_checkpoint",
    "plan_days_at_checkpoint",
    "amount_paid_at_checkpoint",
    "list_price_at_checkpoint",
    "transaction_count",
    "payment_method_count",
    "checkpoint_expire_date",

    # Tenure / demographics (segmentation use, not primary churn features)
    "tenure_days",
    "tenure_bucket",
    "age_cleaned",
    "gender_clean",
    "city",
    "registered_via",

    # Engagement (segmentation use, not primary churn features)
    "active_days",
    "total_listening_secs",
    "total_streams",
    "completion_rate",
    "avg_unique_songs",
    "avg_daily_streams",
    "engagement_bucket"
)

print(f"Final Gold table column count: {len(gold_user_features_train_final.columns)}")
print(f"Final Gold table row count   : {gold_user_features_train_final.count():,}")

gold_user_features_train_final.printSchema()

Final Gold table column count: 27
Final Gold table row count   : 970,960
root
 |-- msno: string (nullable = true)
 |-- is_churn: integer (nullable = true)
 |-- has_feb_checkpoint: integer (nullable = true)
 |-- has_visible_transaction: integer (nullable = false)
 |-- auto_renew_at_checkpoint: integer (nullable = true)
 |-- ever_cancelled: integer (nullable = true)
 |-- ever_zero_paid: integer (nullable = true)
 |-- is_zero_paid_at_checkpoint: integer (nullable = true)
 |-- plan_days_at_checkpoint: integer (nullable = true)
 |-- amount_paid_at_checkpoint: integer (nullable = true)
 |-- list_price_at_checkpoint: integer (nullable = true)
 |-- transaction_count: long (nullable = true)
 |-- payment_method_count: long (nullable = true)
 |-- checkpoint_expire_date: date (nullable = true)
 |-- tenure_days: integer (nullable = true)
 |-- tenure_bucket: string (nullable = true)
 |-- age_cleaned: integer (nullable = true)
 |-- gender_clean: string (nullable = true)
 |-- city: integer (nullable =

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS churn_project.gold")
print("Schema created (or already existed).")

Schema created (or already existed).


In [0]:
(
    gold_user_features_train_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.user_features_train")
)

print(f"Written: {GOLD}.user_features_train  ({gold_user_features_train_final.count():,} rows)")

# Final verification
spark.table(f"{GOLD}.user_features_train").show(5, truncate=False)

Written: churn_project.gold.user_features_train  (970,960 rows)
+--------------------------------------------+--------+------------------+-----------------------+------------------------+--------------+--------------+--------------------------+-----------------------+-------------------------+------------------------+-----------------+--------------------+----------------------+-----------+-------------+-----------+------------+----+--------------+-----------+--------------------+-------------+-------------------+------------------+------------------+-----------------+
|msno                                        |is_churn|has_feb_checkpoint|has_visible_transaction|auto_renew_at_checkpoint|ever_cancelled|ever_zero_paid|is_zero_paid_at_checkpoint|plan_days_at_checkpoint|amount_paid_at_checkpoint|list_price_at_checkpoint|transaction_count|payment_method_count|checkpoint_expire_date|tenure_days|tenure_bucket|age_cleaned|gender_clean|city|registered_via|active_days|total_listening_secs|tot

In [0]:
# Scoring cohort: most recent transaction per user, no leakage concern.
# This applies to ALL users active in March (transactions_score), not
# just a pre-defined labeled population — we don't have labels here.

window_latest_score = Window.partitionBy("msno").orderBy(F.col("transaction_date").desc())

reference_txn_score = (
    df_txn_score
    .withColumn("rn", F.row_number().over(window_latest_score))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

print(f"Scoring cohort users with a transaction: {reference_txn_score.count():,}")

Scoring cohort users with a transaction: 1,197,050


In [0]:
subscription_features_score = (
    reference_txn_score
    .select(
        "msno",
        F.col("is_auto_renew").alias("auto_renew_at_checkpoint"),
        F.col("payment_plan_days").alias("plan_days_at_checkpoint"),
        F.col("actual_amount_paid").alias("amount_paid_at_checkpoint"),
        F.col("plan_list_price").alias("list_price_at_checkpoint"),
        F.when(F.col("actual_amount_paid") == 0, 1).otherwise(0).alias("is_zero_paid_at_checkpoint"),
        F.col("membership_expire_date").alias("checkpoint_expire_date")
    )
)

history_features_score = (
    df_txn_score
    .groupBy("msno")
    .agg(
        F.max("is_cancel").alias("ever_cancelled"),
        F.max(F.when(F.col("actual_amount_paid") == 0, 1).otherwise(0)).alias("ever_zero_paid"),
        F.count("*").alias("transaction_count"),
        F.countDistinct("payment_method_id").alias("payment_method_count")
    )
)

print(f"Subscription features: {subscription_features_score.count():,} users")
print(f"History features      : {history_features_score.count():,} users")

Subscription features: 1,197,050 users
History features      : 1,197,050 users


In [0]:
tenure_features_score = (
    df_members
    .select("msno", "registration_init_time", "city", "gender", "registered_via", "bd")
    .withColumn(
        "tenure_days_raw",
        F.datediff(REFERENCE_DATE_SCORE, F.col("registration_init_time"))
    )
    .withColumn("tenure_days", F.greatest(F.col("tenure_days_raw"), F.lit(0)))
    .withColumn(
        "tenure_bucket",
        F.when(F.col("tenure_days") < 182, "< 6 months")
         .when(F.col("tenure_days") < 365, "6-12 months")
         .when(F.col("tenure_days") < 1095, "1-3 years")
         .when(F.col("tenure_days") < 2190, "3-6 years")
         .otherwise("6+ years")
    )
    .withColumn(
        "age_cleaned",
        F.when((F.col("bd") < 5) | (F.col("bd") > 100), None).otherwise(F.col("bd"))
    )
    .withColumn(
        "gender_clean",
        F.when(F.col("gender").isNull(), "unknown").otherwise(F.col("gender"))
    )
    .drop("tenure_days_raw", "bd", "gender")
)

print(f"Tenure features built (full members table): {tenure_features_score.count():,} users")

Tenure features built (full members table): 6,769,473 users


In [0]:
engagement_features_score = (
    df_logs_score
    .groupBy("msno")
    .agg(
        F.count("*").alias("active_days"),
        F.sum("total_secs").alias("total_listening_secs"),
        F.sum(F.col("num_25") + F.col("num_50") + F.col("num_75") + F.col("num_985") + F.col("num_100")).alias("total_streams"),
        F.sum("num_100").alias("total_completed_streams"),
        F.avg("num_unq").alias("avg_unique_songs")
    )
    .withColumn("avg_daily_streams", F.col("total_streams") / F.col("active_days"))
    .withColumn("avg_listening_secs_per_day", F.col("total_listening_secs") / F.col("active_days"))
    .withColumn("completion_rate", F.col("total_completed_streams") / F.col("total_streams"))
    .withColumn(
        "engagement_bucket",
        F.when(F.col("avg_daily_streams") < 20.60, "Light")
         .when(F.col("avg_daily_streams") < 35.33, "Medium")
         .otherwise("Heavy")
    )
)

print(f"Engagement features built for {engagement_features_score.count():,} users (have log activity)")

Engagement features built for 1,103,894 users (have log activity)


In [0]:
# Base population: users with a transaction in March (eligibility candidates)
gold_user_features_score = (
    reference_txn_score.select("msno")
    .join(subscription_features_score, "msno", "left")
    .join(history_features_score, "msno", "left")
    .join(tenure_features_score, "msno", "left")
    .join(engagement_features_score, "msno", "left")
)

# Zero-impute engagement for users with a transaction but no March logs
# (passive subscribers — same logic as training cohort)
engagement_cols_to_zero_impute = [
    "active_days", "total_listening_secs", "total_streams",
    "total_completed_streams", "avg_daily_streams",
    "avg_listening_secs_per_day", "completion_rate"
]

for c in engagement_cols_to_zero_impute:
    gold_user_features_score = gold_user_features_score.withColumn(
        c, F.when(F.col(c).isNull(), F.lit(0)).otherwise(F.col(c))
    )

gold_user_features_score = gold_user_features_score.withColumn(
    "engagement_bucket",
    F.when(F.col("engagement_bucket").isNull(), F.lit("Light")).otherwise(F.col("engagement_bucket"))
)

print(f"Scoring Gold table: {gold_user_features_score.count():,} rows")
print(f"(Should equal 1,197,050 — all transacting users in March)")

gold_user_features_score.select(
    "msno", "auto_renew_at_checkpoint", "ever_cancelled", "tenure_days", "avg_daily_streams", "engagement_bucket"
).show(5, truncate=False)

Scoring Gold table: 1,197,050 rows
(Should equal 1,197,050 — all transacting users in March)
+--------------------------------------------+------------------------+--------------+-----------+-----------------+-----------------+
|msno                                        |auto_renew_at_checkpoint|ever_cancelled|tenure_days|avg_daily_streams|engagement_bucket|
+--------------------------------------------+------------------------+--------------+-----------+-----------------+-----------------+
|++0+IdHga8fCSioOVpU8K7y4Asw8AveIApVH2r9q9yY=|0                       |0             |2355       |23.84            |Medium           |
|++0wqjjQge1mBBe5r4ciHGKwtF/m322zkra7CK8I+Mw=|1                       |0             |NULL       |0.0              |Light            |
|++4wuAZmfzMBjRHP4vDpTk+jkj9Xam8SW5rNJrGFEsE=|1                       |0             |NULL       |0.0              |Light            |
|++6xEqu4JANaRY4GjEfEFtLtqOvZvYPyP3uk/PW9Ces=|1                       |0             |334        

In [0]:
# Base population: users with a transaction in March (eligibility candidates)
gold_user_features_score = (
    reference_txn_score.select("msno")
    .join(subscription_features_score, "msno", "left")
    .join(history_features_score, "msno", "left")
    .join(tenure_features_score, "msno", "left")
    .join(engagement_features_score, "msno", "left")
)

# Zero-impute engagement for users with a transaction but no March logs
# (passive subscribers — same logic as training cohort)
engagement_cols_to_zero_impute = [
    "active_days", "total_listening_secs", "total_streams",
    "total_completed_streams", "avg_daily_streams",
    "avg_listening_secs_per_day", "completion_rate"
]

for c in engagement_cols_to_zero_impute:
    gold_user_features_score = gold_user_features_score.withColumn(
        c, F.when(F.col(c).isNull(), F.lit(0)).otherwise(F.col(c))
    )

gold_user_features_score = gold_user_features_score.withColumn(
    "engagement_bucket",
    F.when(F.col("engagement_bucket").isNull(), F.lit("Light")).otherwise(F.col("engagement_bucket"))
)

print(f"Scoring Gold table: {gold_user_features_score.count():,} rows")
print(f"(Should equal 1,197,050 — all transacting users in March)")

gold_user_features_score.select(
    "msno", "auto_renew_at_checkpoint", "ever_cancelled", "tenure_days", "avg_daily_streams", "engagement_bucket"
).show(5, truncate=False)

Scoring Gold table: 1,197,050 rows
(Should equal 1,197,050 — all transacting users in March)
+--------------------------------------------+------------------------+--------------+-----------+-----------------+-----------------+
|msno                                        |auto_renew_at_checkpoint|ever_cancelled|tenure_days|avg_daily_streams|engagement_bucket|
+--------------------------------------------+------------------------+--------------+-----------+-----------------+-----------------+
|++8XaEDast796K/DQcP7sSoRQ8JRnA7X5S+oyXEw62E=|1                       |0             |3398       |18.5             |Light            |
|++0+IdHga8fCSioOVpU8K7y4Asw8AveIApVH2r9q9yY=|0                       |0             |2355       |23.84            |Medium           |
|++6xEqu4JANaRY4GjEfEFtLtqOvZvYPyP3uk/PW9Ces=|1                       |0             |334        |6.777777777777778|Light            |
|++0wqjjQge1mBBe5r4ciHGKwtF/m322zkra7CK8I+Mw=|1                       |0             |NULL       

In [0]:
gold_user_features_score_final = gold_user_features_score.select(
    "msno",

    # Subscription features (same set as training, minus has_feb_checkpoint
    # and has_visible_transaction — not applicable here, since base
    # population is already restricted to transacting users)
    "auto_renew_at_checkpoint",
    "ever_cancelled",
    "ever_zero_paid",
    "is_zero_paid_at_checkpoint",
    "plan_days_at_checkpoint",
    "amount_paid_at_checkpoint",
    "list_price_at_checkpoint",
    "transaction_count",
    "payment_method_count",
    "checkpoint_expire_date",

    # Tenure / demographics
    "tenure_days",
    "tenure_bucket",
    "age_cleaned",
    "gender_clean",
    "city",
    "registered_via",

    # Engagement
    "active_days",
    "total_listening_secs",
    "total_streams",
    "completion_rate",
    "avg_unique_songs",
    "avg_daily_streams",
    "engagement_bucket"
)

print(f"Final scoring Gold table column count: {len(gold_user_features_score_final.columns)}")
print(f"Final scoring Gold table row count   : {gold_user_features_score_final.count():,}")

gold_user_features_score_final.printSchema()

Final scoring Gold table column count: 24
Final scoring Gold table row count   : 1,197,050
root
 |-- msno: string (nullable = true)
 |-- auto_renew_at_checkpoint: integer (nullable = true)
 |-- ever_cancelled: integer (nullable = true)
 |-- ever_zero_paid: integer (nullable = true)
 |-- is_zero_paid_at_checkpoint: integer (nullable = true)
 |-- plan_days_at_checkpoint: integer (nullable = true)
 |-- amount_paid_at_checkpoint: integer (nullable = true)
 |-- list_price_at_checkpoint: integer (nullable = true)
 |-- transaction_count: long (nullable = true)
 |-- payment_method_count: long (nullable = true)
 |-- checkpoint_expire_date: date (nullable = true)
 |-- tenure_days: integer (nullable = true)
 |-- tenure_bucket: string (nullable = true)
 |-- age_cleaned: integer (nullable = true)
 |-- gender_clean: string (nullable = true)
 |-- city: integer (nullable = true)
 |-- registered_via: integer (nullable = true)
 |-- active_days: long (nullable = true)
 |-- total_listening_secs: double (n

In [0]:
(
    gold_user_features_score_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{GOLD}.user_features_score")
)

print(f"Written: {GOLD}.user_features_score  ({gold_user_features_score_final.count():,} rows)")

spark.table(f"{GOLD}.user_features_score").show(5, truncate=False)

Written: churn_project.gold.user_features_score  (1,197,050 rows)
+--------------------------------------------+------------------------+--------------+--------------+--------------------------+-----------------------+-------------------------+------------------------+-----------------+--------------------+----------------------+-----------+-------------+-----------+------------+----+--------------+-----------+--------------------+-------------+-------------------+------------------+------------------+-----------------+
|msno                                        |auto_renew_at_checkpoint|ever_cancelled|ever_zero_paid|is_zero_paid_at_checkpoint|plan_days_at_checkpoint|amount_paid_at_checkpoint|list_price_at_checkpoint|transaction_count|payment_method_count|checkpoint_expire_date|tenure_days|tenure_bucket|age_cleaned|gender_clean|city|registered_via|active_days|total_listening_secs|total_streams|completion_rate    |avg_unique_songs  |avg_daily_streams |engagement_bucket|
+-------------

## Step 3 Summary — Feature Engineering Complete

Two Gold Delta tables built using the same feature logic applied to two
different cohorts, with cohort-appropriate reference-point handling.

### `gold.user_features_train` - 970,960 rows
One row per `train_v2` labeled user. Reference point: each user's **Feb-2017
checkpoint transaction** (the one Kaggle used to define the label), with a
documented fallback for users where that checkpoint isn't visible in our
60-day window. This anchoring was necessary to avoid label leakage — using
each user's *latest* transaction would have meant computing features partly
from post-renewal data.

### `gold.user_features_score` - 1,197,050 rows
One row per March-active transacting user. Reference point: each user's most
recent transaction as of March 31 — no leakage concern here, since we're
describing current state for scoring, not reconstructing a past decision.
Base population is restricted to transacting users (not all members, not
log-only users), since eligibility in Step 5 depends on having a
`membership_expire_date` to evaluate.

### Column dictionary (shared schema across both tables)

| Column | Description | Role |
|---|---|---|
| `msno` | User ID | Identifier |
| `is_churn` | Churn label | **Train only** |
| `has_feb_checkpoint` | 1 if user's Feb-expiry transaction is visible in window | **Train only**, data quality flag |
| `has_visible_transaction` | 1 if user has any transaction in window | **Train only** — strongest churn signal found (6.26% vs 84.29%) |
| `auto_renew_at_checkpoint` | Auto-renew status at reference transaction | Primary churn predictor (~3.3x gap) |
| `ever_cancelled` | 1 if any transaction in window has `is_cancel=1` | Primary churn predictor (~3.0x gap) |
| `ever_zero_paid` | 1 if any transaction had `actual_amount_paid=0` | Primary churn predictor (~6x gap) |
| `is_zero_paid_at_checkpoint` | $0 payment specifically at reference transaction | Supporting signal |
| `plan_days_at_checkpoint`, `amount_paid_at_checkpoint`, `list_price_at_checkpoint` | Plan details at reference transaction | Supporting signal (low expected importance — 99.8% plan concentration) |
| `transaction_count`, `payment_method_count` | Transaction frequency/diversity in window | Supporting signal |
| `checkpoint_expire_date` | Expiry date at reference transaction | Used for eligibility filtering in Step 5 |
| `tenure_days`, `tenure_bucket` | Account age from `registration_init_time`, clipped at 0 | **Segmentation use (Step 7)** — not a churn predictor |
| `age_cleaned`, `gender_clean`, `city`, `registered_via` | Demographics | Supporting / segmentation use |
| `active_days`, `total_listening_secs`, `total_streams`, `completion_rate`, `avg_unique_songs`, `avg_daily_streams` | Engagement metrics, zero-imputed for transacting users with no logs | **Segmentation use (Step 7)** — not a churn predictor |
| `engagement_bucket` | Light/Medium/Heavy, percentile-based cutoffs from EDA | **Segmentation use (Step 7)** |

### Key decisions carried forward
- Reference-transaction logic prevents label leakage in the training table
- Subscription mechanics (auto-renew, cancellation, zero-payment, transaction
  visibility) are the features expected to drive the Step 4 model
- Tenure and engagement are retained for **experiment segmentation only**
  (Step 7), per EDA findings that they don't predict churn
- Three null-handling populations documented and handled explicitly, not
  silently: no members record, log-only users (train cohort), fully inactive
  users (train cohort)

**Next:** Step 4 — train a churn model on `gold.user_features_train`,
score `gold.user_features_score`, output `gold.scored_users`.